# Day 3 · 실습 ⑦ — CLIP Zero-shot 이미지 분류

**연계 이론:** Day 3 이론 4(CLIP과 Vision-Language Model)  
**모델:** OpenAI CLIP `ViT-B/32`

### 실습 목표
1. 동일한 검증 이미지의 CLIP Image Embedding 확인
2. `encode_text()`로 클래스 설명을 Text Embedding으로 변환
3. 이미지·텍스트 간 Cosine Similarity 계산
4. 프롬프트 변경에 따른 분류 결과·Heatmap 비교

**50분 운영 예시:** 모델·데이터 확인 10분 → Embedding 계산 10분 → Zero-shot 분류 15분 → 프롬프트 비교 10분 → 정리 5분.

> Zero-shot은 **현재 개미·벌 작업의 정답 레이블로 추가 학습하지 않는다**는 뜻입니다. 사전학습된 CLIP은 그대로 사용하며, 두 개 후보 텍스트 사이의 상대적인 점수를 비교합니다.

## 0. 실행 준비

- 사전 설치: Python, PyTorch, TorchVision, `ftfy`, `regex`, `tqdm`, Pillow, NumPy, Pandas, Matplotlib
- OpenAI 공식 CLIP 설치(터미널): `python -m pip install git+https://github.com/openai/CLIP.git`
- 최초 사용 시 CLIP ViT-B/32 모델 가중치를 다운로드합니다. `DAY3_CLIP_MODEL`에 `.pt` 모델 경로를 지정하면 로컬 체크포인트를 사용할 수 있습니다.
- GPU: RTX 4090 권장, CPU 실행도 가능. 공식 소스: [openai/CLIP](https://github.com/openai/CLIP)
- 개미·벌 이미지 12장을 그대로 사용합니다. 정답 클래스는 **결과 해석**에만 사용하고 CLIP 모델은 추가 학습하지 않습니다.

> 다른 이름의 `clip` 패키지가 설치되어 공식 `clip.load()`가 없는 경우 공식 저장소에서 다시 설치해야 합니다.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
try:
    import clip  # OpenAI 공식 GitHub 저장소의 clip 패키지
except ImportError as exc:
    raise ImportError(
        "OpenAI CLIP을 설치해야 합니다: python -m pip install git+https://github.com/openai/CLIP.git"
    ) from exc
if not hasattr(clip, "load") or not hasattr(clip, "tokenize"):
    raise ImportError("현재 clip 모듈이 OpenAI 공식 CLIP이 아닙니다. 설치 경로를 확인하세요.")

SEED = 42
rng = np.random.default_rng(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLIP_MODEL_SOURCE = os.getenv("DAY3_CLIP_MODEL", "ViT-B/32")
CLIP_CACHE_DIR = os.getenv("DAY3_CLIP_CACHE_DIR")  # 지정하지 않으면 ~/.cache/clip
print("device:", DEVICE, "| model:", CLIP_MODEL_SOURCE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. 이미지 데이터 선택

DINOv3 실습과 동일한 조건: 공식 PyTorch 개미·벌 검증 이미지에서 클래스별 6장씩 고정 선택합니다. 각 이미지의 Label은 결과 비교용입니다.

In [ ]:
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA_ROOT = Path(os.getenv("DAY3_DATA_ROOT", str(Path.cwd() / "data" / "hymenoptera_data"))).expanduser().resolve()
DATA_URL = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"

def ensure_dataset(root=DATA_ROOT):
    required = [root / "train" / "ants", root / "train" / "bees",
                root / "val" / "ants", root / "val" / "bees"]
    if all(p.is_dir() for p in required):
        print(f"데이터셋 준비됨: {root}")
        return root

    # PyTorch 공식 튜토리얼의 공개 개미/벌 데이터셋을 1회 내려받습니다.
    # 공동 실습에서는 강사가 미리 다운로드하여 DAY3_DATA_ROOT를 지정하세요.
    root.parent.mkdir(parents=True, exist_ok=True)
    archive = root.parent / "hymenoptera_data.zip"
    if not archive.is_file():
        print("공식 PyTorch 개미/벌 데이터셋 다운로드 중...")
        tmp_file = archive.with_suffix(".zip.part")
        try:
            with urllib.request.urlopen(DATA_URL, timeout=90) as response, tmp_file.open("wb") as out:
                shutil.copyfileobj(response, out)
            tmp_file.replace(archive)
        finally:
            tmp_file.unlink(missing_ok=True)

    with zipfile.ZipFile(archive) as zf:
        target_base = root.parent.resolve()
        for entry in zf.infolist():
            dest = (target_base / entry.filename).resolve()
            if not dest.is_relative_to(target_base):
                raise ValueError(f"압축 경로가 허용 범위를 벗어났습니다: {entry.filename}")
        zf.extractall(target_base)

    if not all(p.is_dir() for p in required):
        raise FileNotFoundError(f"데이터셋의 train/val/ants/bees 폴더를 찾지 못했습니다: {root}")
    print(f"데이터셋 다운로드 완료: {root}")
    return root

ensure_dataset()

In [ ]:
IMAGES_PER_CLASS = 6
image_paths = []
true_labels = []
for class_name in ("ants", "bees"):
    candidates = sorted((DATA_ROOT / "val" / class_name).glob("*.jpg"))
    if len(candidates) < IMAGES_PER_CLASS:
        raise ValueError(f"{class_name}: 이미지가 {IMAGES_PER_CLASS}장보다 적습니다.")
    choices = rng.choice(len(candidates), size=IMAGES_PER_CLASS, replace=False)
    for idx in choices:
        image_paths.append(candidates[int(idx)])
        true_labels.append(class_name)

images = []
for path in image_paths:
    with Image.open(path) as im:
        images.append(im.convert("RGB"))
short_labels = [f"{name}-{i+1}" for name in ("ants", "bees")
                for i in range(IMAGES_PER_CLASS)]

fig, axes = plt.subplots(2, IMAGES_PER_CLASS, figsize=(15, 5.2))
for ax, im, lbl in zip(axes.flat, images, short_labels):
    ax.imshow(im)
    ax.set_title(lbl)
    ax.axis("off")
plt.suptitle("Fixed validation images for zero-shot evaluation")
plt.tight_layout()
plt.show()

## 2. OpenAI CLIP ViT-B/32 불러오기

`clip.load()`는 모델과 **그 모델에 맞는 전처리**를 함께 반환합니다. DINOv3의 전처리 설정을 CLIP에 그대로 사용하지 않습니다.

In [ ]:
try:
    model, preprocess = clip.load(
        CLIP_MODEL_SOURCE, device=DEVICE, jit=False, download_root=CLIP_CACHE_DIR
    )
except Exception as exc:
    raise RuntimeError(
        "CLIP 모델 로딩 실패: GitHub 패키지 설치, 모델 가중치 다운로드 가능 여부 또는 "
        "DAY3_CLIP_MODEL / DAY3_CLIP_CACHE_DIR 설정을 확인하세요."
    ) from exc
model.eval()
image_inputs = torch.stack([preprocess(im) for im in images]).to(DEVICE)
print("입력 이미지 텐서:", tuple(image_inputs.shape))

## 3. Image Embedding과 Text Embedding

새 Head 학습 없이 CLIP Encoder로 이미지와 텍스트의 Embedding을 만듭니다. Text Encoder에는 클래스 이름만 전달할 수도 있고 자연어 설명 문장을 전달할 수도 있습니다.

In [ ]:
# 현재 분류 작업의 두 후보 클래스
CLASS_NAMES = ["ants", "bees"]
PROMPT_VARIANTS = {
    "A_label_only": ["ant", "bee"],
    "B_photo": ["a photo of an ant", "a photo of a bee"],
    "C_close_up": ["a close-up photo of an ant", "a close-up photo of a bee"],
}

with torch.inference_mode():
    # 공식 CLIP은 CUDA에서 fp16 출력을 낼 수 있으므로 float32로 정규화합니다.
    image_vectors = model.encode_image(image_inputs).float()
    image_vectors = F.normalize(image_vectors, dim=-1)
print("Image Embeddings:", tuple(image_vectors.shape))


def get_text_vectors(prompts):
    tokens = clip.tokenize(prompts).to(DEVICE)
    with torch.inference_mode():
        text_vectors = model.encode_text(tokens).float()
    return F.normalize(text_vectors, dim=-1)

for name, prompts in PROMPT_VARIANTS.items():
    text_vectors = get_text_vectors(prompts)
    print(f"{name:16} | Text Embeddings {tuple(text_vectors.shape)} | {prompts}")

## 4. Zero-shot Classification — 후보 텍스트 유사도 비교

각 이미지의 Embedding과 두 후보 텍스트의 Embedding을 비교해 **가장 높은 Cosine Similarity**를 가진 후보를 선택합니다. 이는 새로운 Head를 학습하지 않은 분류 방식입니다.

**주의:** Cosine Similarity는 정답 확률이 아닙니다. 후보 텍스트를 변경하면 비교 결과도 달라질 수 있습니다.

In [ ]:
# 먼저 B_photo 조건을 기준으로 유사도를 계산합니다.
DEFAULT_PROMPT = "B_photo"
text_vectors = get_text_vectors(PROMPT_VARIANTS[DEFAULT_PROMPT])
cosine_scores = (image_vectors @ text_vectors.T).cpu().numpy()  # [N, 2]
pred_indices = cosine_scores.argmax(axis=1)
pred_labels = [CLASS_NAMES[i] for i in pred_indices]

prediction_df = pd.DataFrame({
    "image": short_labels,
    "true_label": true_labels,
    "cos_ant": cosine_scores[:, 0],
    "cos_bee": cosine_scores[:, 1],
    "prediction": pred_labels,
})
prediction_df["correct"] = prediction_df["true_label"] == prediction_df["prediction"]
display(prediction_df.round(3))
print("두 후보 클래스에 한정한 예시 정확도:",
      f"{prediction_df['correct'].mean():.1%}", "(작은 검증 샘플이므로 일반 성능 아님)")

fig, ax = plt.subplots(figsize=(7, 5))
heatmap = ax.imshow(cosine_scores, vmin=-1, vmax=1, cmap="viridis", aspect="auto")
ax.set_xticks(range(2), PROMPT_VARIANTS[DEFAULT_PROMPT], rotation=10)
ax.set_yticks(range(len(short_labels)), short_labels)
ax.set_title("CLIP Image–Text Cosine Similarity")
fig.colorbar(heatmap, ax=ax, label="Cosine Similarity")
plt.tight_layout()
plt.show()

## 5. 프롬프트 변경 비교(B+C)

**조건 변경 활동:** A(클래스 이름만), B(사진 설명), C(근접 촬영 설명) 간 **텍스트만** 변경합니다. 모델·입력 이미지·후보 클래스 수는 동일합니다.

- 각 프롬프트에서 점수가 어떻게 달라지는지 확인합니다.
- 예측 클래스가 바뀌는 이미지가 있는지 확인합니다.
- 각 조건의 2개 후보 내 정확도는 **관찰용 수치**이며, 다른 작업으로 일반화하지 않습니다.

In [ ]:
all_rows = []
score_by_prompt = {}
for name, prompts in PROMPT_VARIANTS.items():
    text_vecs = get_text_vectors(prompts)
    scores = (image_vectors @ text_vecs.T).cpu().numpy()
    score_by_prompt[name] = scores
    preds = scores.argmax(axis=1)
    for i, pred in enumerate(preds):
        all_rows.append({"prompt_type": name, "image": short_labels[i],
                         "true_label": true_labels[i], "prediction": CLASS_NAMES[pred],
                         "correct": CLASS_NAMES[pred] == true_labels[i],
                         "cos_ant": scores[i, 0], "cos_bee": scores[i, 1]})
all_df = pd.DataFrame(all_rows)
summary_df = (all_df.groupby("prompt_type", as_index=False)
              .agg(num_images=("image", "count"), example_accuracy=("correct", "mean")))
display(summary_df)

pivot = all_df.pivot(index="image", columns="prompt_type", values="prediction")
display(pivot)

# 한 장을 선택해 문구 변경에 따른 유사도를 시각화합니다.
# [학생 변경] 0부터 len(images)-1까지 다른 이미지로 바꿔 보세요.
SAMPLE_INDEX = 0
if not 0 <= SAMPLE_INDEX < len(images):
    raise ValueError("SAMPLE_INDEX가 범위를 벗어났습니다.")
keys = list(PROMPT_VARIANTS)
x = np.arange(len(keys))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - width/2, [score_by_prompt[k][SAMPLE_INDEX, 0] for k in keys],
       width, label="ant")
ax.bar(x + width/2, [score_by_prompt[k][SAMPLE_INDEX, 1] for k in keys],
       width, label="bee")
ax.set_xticks(x, keys)
ax.set_ylabel("Cosine Similarity")
ax.set_title(f"Prompt Comparison: {short_labels[SAMPLE_INDEX]}")
ax.legend()
plt.tight_layout()
plt.show()

## 6. 분류 점수와 확률 해석

Cosine Similarity는 모델이 만든 Embedding 사이의 **유사도**입니다. 이 값을 Softmax로 정규화하면 현재 후보군 내 상대적 분류 점수를 만들 수 있지만, **실제 정답 확률로 보정된 값이라는 보장은 없습니다**.

아래 셀은 첫 번째 이미지에 대해 OpenAI CLIP 모델의 `logit_scale`을 반영한 상대 점수를 확인하는 추가 관찰 코드입니다. 후보 텍스트가 바뀌면 값이 달라질 수 있습니다.

In [ ]:
# CLIP의 학습된 logit_scale과 Softmax를 사용한 후보 내 상대 점수
with torch.inference_mode():
    text_v = get_text_vectors(PROMPT_VARIANTS[DEFAULT_PROMPT])
    logits = model.logit_scale.exp() * (image_vectors[:1] @ text_v.T)
    relative_scores = logits.softmax(dim=-1)[0].cpu().numpy()
print("입력 이미지:", short_labels[0])
for label, score in zip(CLASS_NAMES, relative_scores):
    print(f"{label:>5s}: 후보 내 상대 점수 {score:.3f}")
print("※ 보정된 실제 정답 확률을 뜻하지 않습니다.")

## 7. 결과 저장과 학습 질문

`results/` 폴더에 프롬프트별 유사도·예측·결과 요약을 저장합니다.

In [ ]:
result_dir = Path.cwd() / "results"
result_dir.mkdir(parents=True, exist_ok=True)
all_df.to_csv(result_dir / "day3_04_clip_prompt_predictions.csv",
              index=False, encoding="utf-8-sig")
summary_df.to_csv(result_dir / "day3_04_clip_prompt_summary.csv",
                  index=False, encoding="utf-8-sig")
print("CSV 저장 완료:", result_dir)

**학습 내용 확인**

1. DINOv3에서는 Image–Image를 비교했는데, CLIP에서는 어떤 두 Encoder가 필요한가요?
2. 현재 실습에서 새로운 Classification Head를 학습했나요? 그 이유는 무엇인가요?
3. `encode_image()`와 `encode_text()`의 출력 Shape를 확인해 보세요.
4. 같은 이미지에서 프롬프트를 변경했을 때 어떤 점수가 달라졌나요?
5. Cosine Similarity와 Softmax 점수를 왜 같은 의미로 설명하면 안 되나요?

**다음 학습 연결:** Day 4에서는 이미지 전체의 클래스뿐 아니라 **Bounding Box로 객체 위치**, **Mask로 픽셀 영역**까지 예측하는 Detection·Segmentation으로 확장합니다.